<a href="https://colab.research.google.com/github/KHADIJA2008-KB/flyrank-in-A1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KHADIJA2008-KB/redesigned-guacamole/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Classification, used to produce a ranking.**

I'm framing this as a classification problem: for each page, predict whether it belongs to the
"declining" class or not. But the end deliverable isn't just a yes/no label — I take the model's
predicted probability for each page and use it to *rank* all pages from most to least likely to
be declining. That ranked list is what a content reviewer actually works from. So the underlying
ML task is classification; the way it's consumed is as a ranking.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy):** `is_declining_label`, defined as `trend_direction == "down"`.

This is a proxy, not a true future outcome — it's a bucket calculated from the current window,
not something that happens *after* a decision point. I'm reusing the starter's label at this
stage because it's simple, defensible, and lets me focus on framing before I attempt a stronger
forward-looking label (e.g. prior 90 days -> decline in next 30 days) in a later week.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("/content/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Declining rate:", round(df["is_declining_label"].mean(), 3))
print(df["trend_direction"].value_counts())

Declining rate: 0.542
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50.**

A content team can only realistically review a limited number of pages per cycle — say the top
50 the model flags. Precision@50 asks: of those top 50, how many are actually declining? That
directly matches how the output gets used, unlike overall accuracy, which would be misleading
here since most pages aren't declining. I already used this metric in notebooks 1 and 2, so it's
consistent with how I've been evaluating rules and models so far.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

print("Precision@50 function defined and ready to reuse below.")

Precision@50 function defined and ready to reuse below.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one row = one page.**

`client_id` appears as a column on each page's row (for grouping/validation later), not as the
row itself. Below is a real slice of the dataframe showing this.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

cols = ["content_id", "client_id", "impressions_90d", "days_since_last_update",
        "avg_position", "ctr", "trend_direction", "is_declining_label"]
df[cols].head(10)

,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,down,1
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,0.09,down,1
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,0.16,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

I already tested this directly in notebook 2. A hand-written rule ("stale AND visible") scored
Precision@50 of 0.680. A depth-3 decision tree, trained on the same signals, scored 0.720 on the
same metric — beating the hand rule once the ranking goes deeper than the very top few pages.
The pattern isn't a single clean threshold; it involves a combination of impressions, position,
content age, and CTR interacting in ways that are hard to write as one or two if-statements, but
that a model can pick up from the data itself.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Reference: results already computed in notebook 02
print("Hand rule Precision@50: 0.680")
print("Depth-3 tree Precision@50: 0.720")
print("The tree beats the hand rule at this depth in ranking quality.")

Hand rule Precision@50: 0.680
Depth-3 tree Precision@50: 0.720
The tree beats the hand rule at this depth in ranking quality.


## Self-check

Before you submit, confirm each line honestly:

- [ ✔] Every section above is filled — markdown thinking AND the code that backs it
- [ ✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✔] No client names, URLs, or private queries anywhere
- [ ✔] My claims use careful words: observed, measured, directional, decision-support
- [ ✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.